# Parameter Study Analysis

This notebook imports data collected by a Daphne parameter study, analyzes
scalability and performance metrics, and visualizes results.

## 1. Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dask.dataframe as dd
%matplotlib inline

# Enable copy-on-write to follow pandas recommendation.
# see https://pandas.pydata.org/docs/user_guide/copy_on_write.html
pd.options.mode.copy_on_write = True

In [ ]:
# Load collected data from Parquet file.
# Replace '../output.parquet' with the path to your data file.
# Data is loaded in chunks and processed to categorize columns to limit peak
# memory usage.
delayedData = dd.read_parquet(
    '../output.parquet',
    columns=['timestamp', 'mark', 'type', 'sid', 'NumNodes', 'TxPerSecond']
)

def process_chunk(chunk):
    chunk = dd.from_delayed(chunk).compute()
    for col in ['mark', 'type']:
        chunk[col] = chunk[col].astype('category')
    return chunk

data = pd.concat((process_chunk(chunk) for chunk in delayedData.to_delayed()), ignore_index=True)

## 2. Number of Messages
The following chart illustrates the number of messages being transferred over
the network depending on the number of involved nodes and transaction rates.


In [ ]:
msgsent = data[data['mark'] == 'MsgSent']
grouped = msgsent.groupby(['sid']).size().reset_index(name='Count')

scenarios = data[data['mark'] == 'StudyStarted'][['sid', 'NumNodes', 'TxPerSecond']]

grouped = pd.merge(grouped, scenarios, on='sid')

plt.figure(figsize=(10, 6))
for tx_rate in sorted(grouped['TxPerSecond'].unique()):
    subset = grouped[grouped['TxPerSecond'] == tx_rate]
    plt.plot(subset['NumNodes'], subset['Count'], marker='o', label=f'TxPerSecond={tx_rate}')

plt.title('Total Messages Sent by NumNodes and Transaction Rate')
plt.xlabel('NumNodes')
plt.ylabel('Total Messages Sent')
plt.legend(title='TxPerSecond')
plt.grid(True)
plt.show()

In [ ]:
msgsent = data[data['mark'] == 'MsgSent']
grouped = msgsent.groupby(['sid', 'type'], observed=True).size().reset_index(name='Count')

scenarios = data[data['mark'] == 'StudyStarted'][['sid', 'NumNodes', 'TxPerSecond']]
grouped = pd.merge(grouped, scenarios, on='sid')
grouped = grouped[grouped['TxPerSecond'] == 20]

pivoted = grouped.pivot_table(index=['NumNodes', 'TxPerSecond'], columns='type', values='Count', fill_value=0, observed=True)
pivoted = pivoted.reset_index()

# Plot stacked bars for each TxPerSecond value
plt.figure(figsize=(12, 7))
for tx_rate in sorted(pivoted['TxPerSecond'].unique()):
    subset = pivoted[pivoted['TxPerSecond'] == tx_rate]
    subset.set_index('NumNodes').plot(kind='bar', stacked=True, ax=plt.gca(), legend=True, alpha=0.7)

plt.title('Stacked Bar Chart of Messages Sent by NumNodes and Message Type')
plt.xlabel('NumNodes')
plt.ylabel('Total Messages Sent')
plt.legend(title='Message Type')
plt.grid(True)
plt.show()